In [5]:
import pandas as pd
import altair as alt

# data
df = pd.read_csv("cleaned_rental_data.csv")

# Rename columns
df = df.rename(columns={
    "Total Rent Avg": "Rent",
    "Utility Cost Avg": "Utilities"
})


df["Neighborhood"] = df["Neighborhood"].replace({
    "West End (residential area near the North End and TD Garden)": "West End"
})


melted_df = df.melt(
    id_vars=["Neighborhood"],
    value_vars=["Rent", "Utilities"],
    var_name="Cost Type",
    value_name="Cost"
)


avg_costs = melted_df.groupby(["Neighborhood", "Cost Type"], as_index=False)["Cost"].mean()

# total per neighborhood
pivot_df = avg_costs.pivot(index="Neighborhood", columns="Cost Type", values="Cost").fillna(0)
pivot_df["Total"] = pivot_df["Rent"] + pivot_df["Utilities"]
pivot_df["Rent %"] = pivot_df["Rent"] / pivot_df["Total"] * 100
pivot_df["Utilities %"] = pivot_df["Utilities"] / pivot_df["Total"] * 100


pivot_reset = pivot_df.reset_index()
melted_with_pct = pd.melt(
    pivot_reset,
    id_vars=["Neighborhood", "Total", "Rent %", "Utilities %"],
    value_vars=["Rent", "Utilities"],
    var_name="Cost Type",
    value_name="Cost"
)


melted_with_pct["Percent of Total"] = melted_with_pct.apply(
    lambda row: row["Rent %"] if row["Cost Type"] == "Rent" else row["Utilities %"],
    axis=1
)

# neighborhoods by total cost
sorted_neighborhoods = pivot_df.sort_values("Total").index.tolist()

# color scale
color_scale = alt.Scale(domain=["Rent", "Utilities"], range=["#5a93f2", "#0a357f"])

# stacked bar chart
chart = alt.Chart(melted_with_pct).mark_bar().encode(
    x=alt.X("Neighborhood:N", sort=sorted_neighborhoods, title="Neighborhood"),
    y=alt.Y("Cost:Q", stack="zero", title="Average Monthly Housing Cost"),
    color=alt.Color("Cost Type:N", title="Cost Type", scale=color_scale),
    tooltip=[
        alt.Tooltip("Neighborhood:N"),
        alt.Tooltip("Cost Type:N"),
        alt.Tooltip("Cost:Q", format=",.0f"),
        alt.Tooltip("Percent of Total:Q", format=".1f", title="Percent of Total Cost (%)")
    ]
).properties(
    title="Average Rent and Utility Costs by Neighborhood",
    width=750,
    background="#FFFFFF"
).configure_axisX(
    labelAngle=-40
).configure_title(
    fontSize=18,
    anchor='start',
    color='black'
).configure_view(
    stroke=None
)

chart


alt.Chart(...)

In [6]:
# HTML
chart.save("rent_utilities_by_neighborhood.html")
